# Week 1 Homework – Reinforcement Learning for Data Science Club

**From Prediction to Decision Making: Markov Decision Processes**

**Due**: Before Week 4 live session  
**Goal**: Build intuition for MDPs by implementing core concepts on two gridworld environments.  
**Estimated time**: 30-60 mins 

We will work with two environments:
- **FrozenLake-v1** (non-slippery): 4×4 grid to get familiar
- **A custom 4×4 Gridworld**: Slightly different rewards/transitions for exact solving practice

**Submit**: Run all cells → download .ipynb or export to PDF → upload to [Google Drive / Discord / club platform]

---

## Instructions

Fill in all `# TODO` sections.  
Do **not** change provided function signatures or test cells unless told otherwise.  
We use simple `assert` checks — they must pass for full credit.

Good luck — this notebook directly builds on the Week 1 lecture!

---

## ANSWER KEY

## 0. Setup & Dependencies

In [ ]:
# !pip install gymnasium matplotlib numpy  # uncomment if needed

import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from typing import Callable

%matplotlib inline

## 1. FrozenLake Environment (Non-slippery)

In [ ]:
env = gym.make("FrozenLake-v1", is_slippery=False, render_mode="ansi")

print("Observation space:", env.observation_space)
print("Action space:", env.action_space)
print("\nInitial state:")
print(env.reset())
print(env.render())

### Part 1.1 – Random Policy Rollout

In [ ]:
def random_policy(state: int) -> int:
    """Select action uniformly at random."""
    # SOLUTION: return a random action between 0 and env.action_space.n - 1
    return np.random.randint(0, env.action_space.n)


# Test your random policy (should look random)
print("Sample actions:", [random_policy(0) for _ in range(10)])

In [ ]:
def evaluate_policy(policy: Callable, env, n_episodes=200, max_steps=100, seed=42):
    env = gym.make("FrozenLake-v1", is_slippery=False)  # fresh env
    successes = 0
    for ep in range(n_episodes):
        state, _ = env.reset(seed=seed + ep)
        done = False
        steps = 0
        while not done and steps < max_steps:
            action = policy(state)
            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            steps += 1
        if reward > 0.5:  # reached goal
            successes += 1
    return successes / n_episodes


random_success_rate = evaluate_policy(random_policy, env)
print(f"Random policy success rate: {random_success_rate:.3f}")

assert 0.00 < random_success_rate < 0.20, "Random policy should succeed rarely"

### Part 1.2 – Policy Evaluation (compute V^π)

In [ ]:
def policy_evaluation(policy: Callable, env, gamma=0.99, theta=1e-6):
    """
    Iteratively evaluate the value function V for a given policy.
    Returns: V (numpy array of shape [n_states])
    """
    n_states = env.observation_space.n
    V = np.zeros(n_states)

    while True:
        delta = 0
        for s in range(n_states):
            v_old = V[s]
            # SOLUTION: compute new V[s] = expected [r + gamma * V[s']]
            # Use env.P[s][a] which is list of (prob, next_state, reward, done)
            # Remember: for terminal states, V[s] should stay 0
            
            # Check if state is terminal (hole or goal)
            # In FrozenLake, terminal states have no transitions
            if len(env.P[s]) == 0:
                V[s] = 0.0
            else:
                # Compute expected value under policy π
                v_new = 0.0
                for a in range(env.action_space.n):
                    action_prob = 1.0 / env.action_space.n  # uniform random policy
                    transitions = env.P[s][a]
                    for prob, next_state, reward, done in transitions:
                        if done:
                            # Terminal state: value is just the reward
                            v_new += action_prob * prob * reward
                        else:
                            # Non-terminal: r + gamma * V[s']
                            v_new += action_prob * prob * (reward + gamma * V[next_state])
                V[s] = v_new
            
            delta = max(delta, abs(v_old - V[s]))
        if delta < theta:
            break

    return V


# Evaluate random policy
V_random = policy_evaluation(random_policy, env)
print("Value function for random policy (4×4):")
print(V_random.reshape(4, 4).round(3))

In [ ]:
# Visualize
plt.figure(figsize=(6,5))
plt.imshow(V_random.reshape(4,4), cmap='viridis')
plt.colorbar(label='Value')
plt.title("V^π (random policy) on FrozenLake")
plt.xticks(range(4)); plt.yticks(range(4))
plt.show()

### Part 1.3 – Greedy Policy from V^π

In [ ]:
def get_greedy_policy(V: np.ndarray, env, gamma=0.99):
    """Extract deterministic greedy policy from value function."""
    n_states = len(V)
    policy = np.zeros(n_states, dtype=int)

    for s in range(n_states):
        q_values = []
        for a in range(env.action_space.n):
            # SOLUTION: compute Q(s,a) = expected [r + gamma * V[s']]
            q = 0.0
            transitions = env.P[s][a]
            for prob, next_state, reward, done in transitions:
                if done:
                    # Terminal state: Q(s,a) = reward
                    q += prob * reward
                else:
                    # Non-terminal: Q(s,a) = r + gamma * V[s']
                    q += prob * (reward + gamma * V[next_state])
            q_values.append(q)
        policy[s] = np.argmax(q_values)

    return policy


policy_greedy_random = get_greedy_policy(V_random, env)
print("Greedy policy from random V (action per state):")
print(policy_greedy_random.reshape(4,4))

In [ ]:
greedy_success_rate = evaluate_policy(
    lambda s: policy_greedy_random[s], env
)
print(f"Greedy-from-random success rate: {greedy_success_rate:.3f}")

assert greedy_success_rate > random_success_rate, \
    "Greedy policy should be better than random"

## 2. Value Iteration – Solve FrozenLake Exactly

In [ ]:
def value_iteration(env, gamma=0.99, theta=1e-6):
    n_states = env.observation_space.n
    V = np.zeros(n_states)

    while True:
        delta = 0
        for s in range(n_states):
            v_old = V[s]
            # SOLUTION: V[s] = max_a expected [r + gamma V[s']]
            # For each action, compute Q(s,a), then take max
            
            # Check if terminal state
            if len(env.P[s]) == 0:
                V[s] = 0.0
            else:
                max_q = float('-inf')
                for a in range(env.action_space.n):
                    q = 0.0
                    transitions = env.P[s][a]
                    for prob, next_state, reward, done in transitions:
                        if done:
                            q += prob * reward
                        else:
                            q += prob * (reward + gamma * V[next_state])
                    max_q = max(max_q, q)
                V[s] = max_q
            
            delta = max(delta, abs(v_old - V[s]))
        if delta < theta:
            break

    return V


V_opt = value_iteration(env)
print("Optimal V* (4×4):")
print(V_opt.reshape(4,4).round(3))

In [ ]:
# Extract optimal policy
policy_opt = get_greedy_policy(V_opt, env)
print("Optimal policy (0=←, 1=↓, 2=→, 3=↑):")
print(policy_opt.reshape(4,4))

In [ ]:
# Should be close to 1.0
opt_success_rate = evaluate_policy(
    lambda s: policy_opt[s], env, n_episodes=50
)
print(f"Optimal policy success rate: {opt_success_rate:.3f}")

assert V_opt[15] > 0.999, "Goal state should have value ≈ 1.0"
assert opt_success_rate > 0.95, "Optimal policy should almost always succeed"

## 3. Reflection Questions (short answer)

**Question 1** (2–4 sentences):  
Why does the optimal value function give non-zero values to some frozen tiles even though they give 0 immediate reward?

**Answer:**

The optimal value function assigns non-zero values to frozen tiles because the value function represents the expected future discounted reward from that state, not just the immediate reward. Even though frozen tiles give 0 immediate reward, they have value because they are on the path to the goal state (which gives reward 1.0). The value captures the probability of eventually reaching the goal multiplied by the discounted reward. States closer to the goal or on optimal paths will have higher values because they have a higher probability of leading to the goal, even if their immediate reward is zero.

**Question 2** (2–4 sentences):  
Compare the success rates: random vs. greedy-from-random vs. optimal.  
What does this tell you about the importance of **planning** (using value functions) vs. pure exploration?

**Answer:**

The random policy has a very low success rate (typically < 20%) because it explores without any guidance. The greedy-from-random policy improves significantly because it uses the value function from the random policy to make better decisions, even though that value function wasn't optimal. The optimal policy achieves near-perfect success (typically > 95%) because it uses the optimal value function computed through planning (value iteration). This demonstrates that planning—computing value functions that account for future rewards—is crucial for good performance. Pure exploration (random policy) is inefficient, while planning allows the agent to reason about long-term consequences and find optimal paths to the goal.

**Question 3** (optional bonus):  
If we made the lake slippery again (`is_slippery=True`), how do you think the optimal policy and values would change? Why?

**Answer:**

If we made the lake slippery again, the optimal policy and values would change significantly. The values would generally be lower because the stochastic transitions reduce the probability of successfully reaching the goal—even when taking the optimal action, there's a chance of slipping into an unintended state (possibly a hole). The optimal policy might also change because actions that were previously optimal might now have lower expected values due to the risk of slipping. The agent would need to account for the uncertainty in transitions, potentially choosing actions that are more robust to stochasticity. The value function would reflect this uncertainty by discounting future rewards more heavily due to the increased risk of failure.

---

## Congratulations!

You have now:
- Implemented policy evaluation
- Extracted greedy policies
- Solved an MDP exactly with value iteration

Next week (Tabular RL) we drop the known transition model `P` and learn by interacting — see you then!